In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# Install NLTK
!pip install nltk

import pandas as pd
import nltk
import re

# Download required NLTK resources
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

print("All libraries loaded successfully")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


All libraries loaded successfully


In [4]:
df = pd.read_csv("/content/drive/MyDrive/FactLens_Group9/data/df_clean.csv")

print(f"Dataset loaded: {len(df)} articles")
print(df["label"].value_counts())

Dataset loaded: 38646 articles
label
REAL    21191
FAKE    17455
Name: count, dtype: int64


In [5]:
# Initialize lemmatizer and stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def clean_text(text):
    # 1 - lowercase
    text = text.lower()

    # 2 - remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # 3 - remove punctuation and special characters
    text = re.sub(r"[^a-z\s]", "", text)

    # 4 - remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # 5 - tokenize (split into individual words)
    words = text.split()

    # 6 - remove stopwords and lemmatize
    words = [lemmatizer.lemmatize(word) for word in words
             if word not in stop_words]

    # 7 - join back into a single string
    return " ".join(words)

print("Cleaning function defined successfully")

Cleaning function defined successfully


In [6]:
# Test on a raw fake article
sample = df["text"].iloc[0]
print("BEFORE CLEANING:")
print(sample[:300])

print("\nAFTER CLEANING:")
print(clean_text(sample[:300]))

BEFORE CLEANING:
Donald Trump just couldn t wish all Americans a Happy New Year and leave it at that. Instead, he had to give a shout out to his enemies, haters and  the very dishonest fake news media.  The former reality show star had just one job to do and he couldn t do it. As our Country rapidly grows stronger a

AFTER CLEANING:
donald trump wish american happy new year leave instead give shout enemy hater dishonest fake news medium former reality show star one job country rapidly grows stronger


In [7]:
print("Cleaning text column — this will take 1-2 minutes...")

df["cleaned_text"] = df["text"].apply(clean_text)

print("Done")
print(f"\nSample cleaned article:")
print(df["cleaned_text"].iloc[0][:300])

Cleaning text column — this will take 1-2 minutes...
Done

Sample cleaned article:
donald trump wish american happy new year leave instead give shout enemy hater dishonest fake news medium former reality show star one job country rapidly grows stronger smarter want wish friend supporter enemy hater even dishonest fake news medium happy healthy new year president angry pant tweeted


In [8]:
# Check nothing went wrong
print("Articles before cleaning:", len(df))
print("Empty articles after cleaning:", df["cleaned_text"].apply(lambda x: len(x.strip()) == 0).sum())

# Compare average word count before and after
df["word_count_before"] = df["text"].apply(lambda x: len(str(x).split()))
df["word_count_after"] = df["cleaned_text"].apply(lambda x: len(str(x).split()))

print(f"\nAvg words BEFORE cleaning: {df['word_count_before'].mean():.0f}")
print(f"Avg words AFTER cleaning:  {df['word_count_after'].mean():.0f}")
print(f"Average words removed:     {(df['word_count_before'] - df['word_count_after']).mean():.0f}")

Articles before cleaning: 38646
Empty articles after cleaning: 56

Avg words BEFORE cleaning: 403
Avg words AFTER cleaning:  228
Average words removed:     175


In [9]:
# Check what these empty articles look like before removing
empty = df[df["cleaned_text"].apply(lambda x: len(x.strip()) == 0)]
print("Empty articles label distribution:")
print(empty["label"].value_counts())

print("\nSample original text from empty articles:")
print(df.loc[empty.index, "text"].iloc[0][:300])

Empty articles label distribution:
label
FAKE    56
Name: count, dtype: int64

Sample original text from empty articles:
https://100percentfedup.com/served-roy-moore-vietnamletter-veteran-sets-record-straight-honorable-decent-respectable-patriotic-commander-soldier/


In [10]:
# Remove empty articles
df = df[df["cleaned_text"].apply(lambda x: len(x.strip()) > 0)]

print(f"Articles remaining: {len(df)}")
print(f"\nLabel distribution:")
print(df["label"].value_counts())

Articles remaining: 38590

Label distribution:
label
REAL    21191
FAKE    17399
Name: count, dtype: int64


In [11]:
df.to_csv("/content/drive/MyDrive/FactLens_Group9/data/df_cleaned.csv", index=False)
print("Saved successfully")
print(f"Total articles saved: {len(df)}")

Saved successfully
Total articles saved: 38590
